# ClaimBuster Claim-Detector — Training (DebateIQ Pakistan)

Trains a binary **check-worthy claim** classifier to replace the hand-coded
heuristic in the app. Output: a saved model you load in the backend.

**Setup on Kaggle:**
1. Runtime: **GPU T4** (Settings -> Accelerator).
2. **Add Data** (right panel) -> search **"claimbuster"** -> add a ClaimBuster
   dataset (it has sentences labelled NFS / UFS / CFS, where CFS = check-worthy).
3. Run all cells. The trained model lands in `/kaggle/working/claimbuster-detector`.

In [ ]:
# 1. Install extras (Kaggle already has torch + transformers)
!pip install -q datasets evaluate
import glob, numpy as np, pandas as pd

In [ ]:
# 2. Find the ClaimBuster CSV and map labels to binary (check-worthy = 1)
csvs = glob.glob('/kaggle/input/**/*.csv', recursive=True)
print('CSV files found:'); [print(' ', c) for c in csvs]
assert csvs, 'No CSV found. Use Add Data -> search claimbuster and add a dataset.'

TEXT_COLS  = ['Text','text','Sentence','sentence','claim','Claim']
LABEL_COLS = ['Verdict','verdict','label','Label','Class','class']

def load_claimbuster():
    for path in csvs:
        try: df = pd.read_csv(path)
        except Exception: continue
        tcol = next((c for c in TEXT_COLS  if c in df.columns), None)
        lcol = next((c for c in LABEL_COLS if c in df.columns), None)
        if tcol and lcol:
            print(f'Using {path}  (text="{tcol}", label="{lcol}")')
            return df, tcol, lcol
    raise ValueError('Could not find text+label columns. Inspect the CSV headers above.')

df, tcol, lcol = load_claimbuster()

def to_binary(v):
    s = str(v).strip().upper()
    if s in ('CFS', 'CHECK-WORTHY', 'CHECKWORTHY'): return 1
    try: return 1 if int(float(s)) == 1 else 0   # CFS encoded as 1
    except ValueError: return 0

df['label'] = df[lcol].apply(to_binary)
df = df[[tcol, 'label']].rename(columns={tcol: 'text'}).dropna()
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 5].drop_duplicates('text').reset_index(drop=True)
print('\nTotal:', len(df), '| check-worthy:', int(df.label.sum()), '| not:', int((df.label==0).sum()))
df.head()

In [ ]:
# 3. Split + tokenize
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

tr, va = train_test_split(df, test_size=0.1, stratify=df['label'], random_state=42)
CKPT = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(CKPT)

def tk(b): return tok(b['text'], truncation=True, max_length=128)
ds_tr = Dataset.from_pandas(tr[['text','label']], preserve_index=False).map(tk, batched=True)
ds_va = Dataset.from_pandas(va[['text','label']], preserve_index=False).map(tk, batched=True)
print('train', len(ds_tr), 'val', len(ds_va))

In [ ]:
# 4. Train (distilbert, 2 epochs ~ a few minutes on T4)
import evaluate
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)

model = AutoModelForSequenceClassification.from_pretrained(
    CKPT, num_labels=2,
    id2label={0:'not_checkworthy', 1:'checkworthy'},
    label2id={'not_checkworthy':0, 'checkworthy':1})

acc, f1 = evaluate.load('accuracy'), evaluate.load('f1')
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': acc.compute(predictions=preds, references=p.label_ids)['accuracy'],
            'f1': f1.compute(predictions=preds, references=p.label_ids)['f1']}

args = TrainingArguments(output_dir='out', num_train_epochs=2,
        per_device_train_batch_size=16, learning_rate=2e-5,
        weight_decay=0.01, logging_steps=50, save_strategy='no', report_to=[])

trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                  tokenizer=tok, data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=compute_metrics)
trainer.train()
print('\nFinal eval:', trainer.evaluate())

In [ ]:
# 5. Save the model (this folder is downloadable from Kaggle output)
OUT = '/kaggle/working/claimbuster-detector'
model.save_pretrained(OUT); tok.save_pretrained(OUT)
print('Saved to', OUT)
!ls -la /kaggle/working/claimbuster-detector

In [ ]:
# 6. Sanity check — does it catch puffery the heuristic missed?
from transformers import pipeline
clf = pipeline('text-classification', model=OUT, tokenizer=tok)
tests = ['Forman is the best university in the world.',
         'I had coffee this morning.',
         'The World Bank reported 70% youth unemployment.',
         'Thank you all for coming today.',
         'Pakistan won the cricket world cup in 1992.']
for s in tests:
    r = clf(s)[0]
    print(f'{r["label"]:18s} {r["score"]:.2f}  | {s}')

In [ ]:
# 7. (OPTIONAL) Push to the Hugging Face Hub so the backend can load it by name.
# from huggingface_hub import login
# login(token='hf_XXXXXXXX')            # your HF write token
# REPO = 'your-username/claimbuster-detector'
# model.push_to_hub(REPO); tok.push_to_hub(REPO)
# print('Pushed to', REPO)

## Using the trained model in the backend
Two ways to point the app at it (handled by `backend/src/models.py`):

- **Hub:** push it (cell 7), then set an env var before starting the backend:
  `CLAIM_MODEL=your-username/claimbuster-detector`
- **Local:** download the `claimbuster-detector` folder from Kaggle output, drop it
  into `backend/`, and set `CLAIM_MODEL=./claimbuster-detector`

If `CLAIM_MODEL` is unset, the app keeps using the hand-coded heuristic — so the
trained model is a drop-in upgrade, not a requirement.